# Advanced Problems: A Simple Function Timer

Practice advanced function timing, argument forwarding, keyword-only arguments, `*args`, `**kwargs`, decorators, generators, and benchmark reliability.


## Setup

In [1]:
import time
from functools import wraps
from statistics import mean, median


## Problem 1 — Build a Robust `time_it`

Write `time_it(fn, *args, rep=5, **kwargs)`.

Requirements:

- `fn` is the function to time.
- `*args` are forwarded to `fn`.
- `**kwargs` are forwarded to `fn`.
- `rep` is keyword-only and controls how many times `fn` runs.
- Return the average runtime.
- Reject `rep < 1`.
- Reject non-callable `fn`.

In [2]:
def time_it(fn, *args, rep=5, **kwargs):
    if not callable(fn):
        raise TypeError("fn must be callable")
    if rep < 1:
        raise ValueError("rep must be at least 1")

    start = time.perf_counter()
    for _ in range(rep):
        fn(*args, **kwargs)
    end = time.perf_counter()

    return (end - start) / rep


def add(a, b):
    return a + b


result = time_it(add, 10, 20, rep=10)
assert isinstance(result, float)
assert result >= 0

try:
    time_it(add, 1, 2, rep=0)
except ValueError as ex:
    assert "rep" in str(ex)
else:
    raise AssertionError("Expected ValueError for rep=0")

try:
    time_it(123, rep=1)
except TypeError as ex:
    assert "callable" in str(ex)
else:
    raise AssertionError("Expected TypeError for non-callable fn")

print("Problem 1 passed")

Problem 1 passed


## Problem 2 — Confirm Argument Forwarding

Create a function `capture_call(*args, **kwargs)` that returns exactly what it received.

Then use `time_it` to verify that positional and keyword arguments are correctly forwarded.

Because `time_it` returns timing information, use a side effect to inspect the forwarded call.

In [3]:
calls = []


def capture_call(*args, **kwargs):
    calls.append((args, kwargs))
    return args, kwargs


time_it(capture_call, 1, 2, 3, sep="-", end="!", rep=3)

assert len(calls) == 3
assert calls[0] == ((1, 2, 3), {"sep": "-", "end": "!"})
assert calls[1] == ((1, 2, 3), {"sep": "-", "end": "!"})
assert calls[2] == ((1, 2, 3), {"sep": "-", "end": "!"})

print("Problem 2 passed")

Problem 2 passed


## Problem 3 — Return Both Runtime and Function Result

The original `time_it` discards the function result.

Write `time_it_with_result(fn, *args, rep=5, **kwargs)`.

Requirements:

- Run the function `rep` times.
- Return a dictionary containing:
  - `'average'`
  - `'last_result'`
  - `'rep'`
- Preserve argument forwarding.
- Reject invalid `rep` values.

In [4]:
def time_it_with_result(fn, *args, rep=5, **kwargs):
    if not callable(fn):
        raise TypeError("fn must be callable")
    if rep < 1:
        raise ValueError("rep must be at least 1")

    last_result = None
    start = time.perf_counter()

    for _ in range(rep):
        last_result = fn(*args, **kwargs)

    end = time.perf_counter()

    return {
        "average": (end - start) / rep,
        "last_result": last_result,
        "rep": rep,
    }


def power_list(n, *, start=1, end):
    return [n ** i for i in range(start, end)]


timed = time_it_with_result(power_list, 2, start=1, end=5, rep=3)

assert timed["rep"] == 3
assert timed["last_result"] == [2, 4, 8, 16]
assert isinstance(timed["average"], float)
assert timed["average"] >= 0

print("Problem 3 passed")

Problem 3 passed


## Problem 4 — Avoid the Generator Timing Trap

A generator expression can look extremely fast because creating the generator does not consume it.

Write `time_consumed(fn, *args, consume=None, rep=5, **kwargs)`.

Requirements:

- If `consume is None`, just call `fn`.
- If `consume` is provided, call `consume(fn(*args, **kwargs))`.
- Return the average runtime.
- This allows fair timing of generator-producing functions.

In [5]:
def time_consumed(fn, *args, consume=None, rep=5, **kwargs):
    if rep < 1:
        raise ValueError("rep must be at least 1")

    start = time.perf_counter()

    for _ in range(rep):
        result = fn(*args, **kwargs)
        if consume is not None:
            consume(result)

    end = time.perf_counter()
    return (end - start) / rep


def powers_list(n, *, start=1, end):
    return [n ** i for i in range(start, end)]


def powers_generator(n, *, start=1, end):
    return (n ** i for i in range(start, end))


list_time = time_consumed(powers_list, 2, end=1000, rep=3)
generator_creation_time = time_consumed(powers_generator, 2, end=1000, rep=3)
generator_consumed_time = time_consumed(powers_generator, 2, end=1000, consume=list, rep=3)

assert list_time >= 0
assert generator_creation_time >= 0
assert generator_consumed_time >= 0

print("Problem 4 passed")

Problem 4 passed


## Problem 5 — Collect Individual Timing Samples

Average timing can hide outliers.

Write `time_samples(fn, *args, rep=5, **kwargs)`.

Requirements:

- Return a list of individual runtimes.
- Each runtime should measure exactly one call.
- Reject `rep < 1`.
- Do not print inside the timing function.

In [6]:
def time_samples(fn, *args, rep=5, **kwargs):
    if rep < 1:
        raise ValueError("rep must be at least 1")

    samples = []

    for _ in range(rep):
        start = time.perf_counter()
        fn(*args, **kwargs)
        end = time.perf_counter()
        samples.append(end - start)

    return samples


samples = time_samples(sum, [1, 2, 3, 4, 5], rep=7)

assert len(samples) == 7
assert all(isinstance(sample, float) for sample in samples)
assert all(sample >= 0 for sample in samples)

print("Problem 5 passed")

Problem 5 passed


## Problem 6 — Summarize Benchmark Results

Write `benchmark(fn, *args, rep=10, **kwargs)`.

Requirements:

- Use `time_samples` internally.
- Return a dictionary containing:
  - `'min'`
  - `'max'`
  - `'mean'`
  - `'median'`
  - `'samples'`
- Do not mutate the samples after storing them.

In [7]:
def benchmark(fn, *args, rep=10, **kwargs):
    samples = time_samples(fn, *args, rep=rep, **kwargs)

    return {
        "min": min(samples),
        "max": max(samples),
        "mean": mean(samples),
        "median": median(samples),
        "samples": samples,
    }


stats = benchmark(sorted, [3, 1, 2], rep=5)

assert set(stats) == {"min", "max", "mean", "median", "samples"}
assert len(stats["samples"]) == 5
assert stats["min"] <= stats["mean"] <= stats["max"]
assert stats["min"] <= stats["median"] <= stats["max"]

print("Problem 6 passed")

Problem 6 passed


## Problem 7 — Create a Timing Decorator

Write a decorator `timed`.

Requirements:

- The decorated function should behave like the original function.
- The wrapper should return a dictionary containing:
  - `'result'`
  - `'elapsed'`
- Use `functools.wraps`.
- Support arbitrary positional and keyword arguments.

In [8]:
def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        end = time.perf_counter()

        return {
            "result": result,
            "elapsed": end - start,
        }

    return wrapper


@timed
def multiply(a, b, *, scale=1):
    return a * b * scale


decorated_result = multiply(3, 4, scale=10)

assert decorated_result["result"] == 120
assert decorated_result["elapsed"] >= 0
assert multiply.__name__ == "multiply"

print("Problem 7 passed")

Problem 7 passed


## Problem 8 — Create a Parameterized Timing Decorator

Write `timed_repeat(rep=5)`.

Requirements:

- It should be a decorator factory.
- The decorated function should run `rep` times.
- Return a dictionary containing:
  - `'last_result'`
  - `'average'`
  - `'rep'`
- Reject `rep < 1` when the decorator is created.

In [9]:
def timed_repeat(rep=5):
    if rep < 1:
        raise ValueError("rep must be at least 1")

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            last_result = None
            start = time.perf_counter()

            for _ in range(rep):
                last_result = fn(*args, **kwargs)

            end = time.perf_counter()

            return {
                "last_result": last_result,
                "average": (end - start) / rep,
                "rep": rep,
            }

        return wrapper

    return decorator


@timed_repeat(rep=4)
def subtract(a, b):
    return a - b


repeat_result = subtract(10, 3)

assert repeat_result["last_result"] == 7
assert repeat_result["rep"] == 4
assert repeat_result["average"] >= 0
assert subtract.__name__ == "subtract"

try:
    timed_repeat(rep=0)
except ValueError as ex:
    assert "rep" in str(ex)
else:
    raise AssertionError("Expected ValueError for rep=0")

print("Problem 8 passed")

Problem 8 passed


## Problem 9 — Compare Multiple Functions Fairly

Write `compare(functions, *args, rep=5, consume=None, **kwargs)`.

Requirements:

- `functions` is a dictionary mapping names to functions.
- Time every function with the same arguments.
- If `consume` is provided, consume each function result before stopping the timer.
- Return a dictionary mapping function names to average runtimes.
- This allows fair comparison between list-producing and generator-producing functions.

In [10]:
def compare(functions, *args, rep=5, consume=None, **kwargs):
    if not isinstance(functions, dict):
        raise TypeError("functions must be a dictionary")

    results = {}

    for name, fn in functions.items():
        results[name] = time_consumed(fn, *args, consume=consume, rep=rep, **kwargs)

    return results


functions = {
    "list": powers_list,
    "generator_consumed": powers_generator,
}

comparison = compare(functions, 2, end=500, rep=3, consume=list)

assert set(comparison) == {"list", "generator_consumed"}
assert all(value >= 0 for value in comparison.values())

print("Problem 9 passed")

Problem 9 passed


## Problem 10 — Add Optional Warmup Runs

Benchmark results can be noisy. Sometimes we want warmup runs that are not included in the final timing.

Write `time_with_warmup(fn, *args, warmup=1, rep=5, **kwargs)`.

Requirements:

- Run `fn` `warmup` times before measuring.
- Run `fn` `rep` times while measuring.
- Return the average measured runtime.
- `warmup` and `rep` must be keyword-only.
- Reject `warmup < 0` and `rep < 1`.

In [11]:
def time_with_warmup(fn, *args, warmup=1, rep=5, **kwargs):
    if warmup < 0:
        raise ValueError("warmup must be at least 0")
    if rep < 1:
        raise ValueError("rep must be at least 1")

    for _ in range(warmup):
        fn(*args, **kwargs)

    start = time.perf_counter()

    for _ in range(rep):
        fn(*args, **kwargs)

    end = time.perf_counter()

    return (end - start) / rep


warmup_time = time_with_warmup(sorted, [5, 1, 3, 2, 4], warmup=2, rep=5)
assert warmup_time >= 0

try:
    time_with_warmup(sorted, [1, 2, 3], warmup=-1)
except ValueError as ex:
    assert "warmup" in str(ex)
else:
    raise AssertionError("Expected ValueError for negative warmup")

print("Problem 10 passed")

Problem 10 passed


## Final Best-Practice Notes

When writing timing utilities:

1. Use `time.perf_counter()` for short duration measurements.
2. Make timer configuration arguments keyword-only.
3. Forward user function arguments with `*args` and `**kwargs`.
4. Validate repetition counts.
5. Avoid printing inside benchmark functions.
6. Be careful when timing generators: creation is not the same as consumption.
7. Use multiple samples when timing noisy operations.
8. Preserve decorated function metadata with `functools.wraps`.
